# Variational Inference

In [ ]:
%pip install numpy matplotlib scipy scikit-learn pymc arviz jax numpyro

## Motivation

In the MCMC lectures, we developed sampling-based methods for Bayesian inference. Given a posterior $p(\boldsymbol{\theta} | \mathbf{y}) \propto p(\mathbf{y} | \boldsymbol{\theta}) p(\boldsymbol{\theta})$, Metropolis-Hastings and Gibbs sampling construct Markov chains whose stationary distribution is the posterior. These methods are asymptotically exact: given enough samples, the empirical distribution of the chain converges to the true posterior.

However, MCMC has practical limitations. Each iteration requires evaluating the likelihood for the entire dataset, making it slow for large $n$. Convergence diagnostics are imperfect, and we may not know when the chain has mixed. For complex models with many latent variables, mixing can be extremely slow.

**Variational inference (VI)** takes a fundamentally different approach. Instead of sampling from the posterior, it *approximates* the posterior by solving an optimization problem. We posit a family of tractable distributions $\mathcal{Q}$ and find the member $q^*(\boldsymbol{\theta}) \in \mathcal{Q}$ that is closest to the true posterior:

$$q^*(\boldsymbol{\theta}) = \arg\min_{q \in \mathcal{Q}} \text{KL}(q(\boldsymbol{\theta}) \| p(\boldsymbol{\theta} | \mathbf{y}))$$

This converts an inference problem into an optimization problem. Optimization is typically faster than sampling, and the tools of optimization (convergence monitoring via the objective function, stochastic gradient methods for scalability) apply directly.

The tradeoff is accuracy: VI provides an approximation whose quality depends on the expressiveness of the family $\mathcal{Q}$. MCMC is asymptotically exact but slow. VI is fast but approximate.

### Running Example: Bayesian Logistic Regression

We will use the same Bayesian logistic regression model from the MCMC lecture as a running example, so we can directly compare the two approaches.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from scipy.optimize import minimize

np.random.seed(42)

# Simulate logistic regression data
n = 200
p = 2
X = np.column_stack([np.ones(n), np.random.normal(0, 1, n)])
beta_true = np.array([0.5, 1.5])
prob = 1 / (1 + np.exp(-X @ beta_true))
y = np.random.binomial(1, prob)

print(f"Simulated data: n={n}, p={p}")
print(f"True coefficients: beta = {beta_true}")
print(f"Observed proportion of y=1: {np.mean(y):.3f}")

In [ ]:
def log_posterior(beta, X, y, tau=10.0):
    """Log-posterior for Bayesian logistic regression (up to a constant)."""
    eta = X @ beta
    log_lik = np.sum(y * eta - np.logaddexp(0, eta))
    log_prior = -0.5 * np.sum(beta**2) / tau**2
    return log_lik + log_prior


# MAP estimate for reference
res = minimize(lambda b: -log_posterior(b, X, y), x0=np.zeros(p), method="BFGS")
beta_map = res.x
print(f"MAP estimate: beta = {beta_map}")

## KL Divergence

Before deriving variational inference, we need to understand the distance measure it uses: the Kullback-Leibler (KL) divergence.

### Definition

The KL divergence from distribution $q$ to distribution $p$ is:

$$\text{KL}(q \| p) = \int q(x) \log \frac{q(x)}{p(x)} \, dx = E_q\left[\log \frac{q(X)}{p(X)}\right]$$

KL divergence has these properties:

- **Non-negative:** $\text{KL}(q \| p) \geq 0$, with equality if and only if $q = p$ almost everywhere. This follows from Jensen's inequality applied to $-\log$.
- **Not symmetric:** $\text{KL}(q \| p) \neq \text{KL}(p \| q)$ in general, so it is not a true distance metric.
- **Requires absolute continuity:** $q$ must be zero wherever $p$ is zero (the support of $q$ must be contained in the support of $p$).

### Directionality Matters

The asymmetry of KL divergence has important practical consequences. The two directions lead to very different approximations:

**Reverse KL: $\text{KL}(q \| p)$** (what VI minimizes)

This is called **zero-forcing** or **mode-seeking**. Wherever $p(\boldsymbol{\theta} | \mathbf{y})$ is near zero, $q(\boldsymbol{\theta})$ must also be near zero (otherwise $\log(q/p) \to \infty$ and the integral blows up). This forces $q$ to concentrate where $p$ has mass. If $p$ is multimodal, $q$ will typically lock onto a single mode rather than spreading mass across all modes.

**Forward KL: $\text{KL}(p \| q)$** (what expectation propagation minimizes)

This is called **zero-avoiding** or **mass-covering**. Wherever $p(\boldsymbol{\theta} | \mathbf{y})$ has mass, $q(\boldsymbol{\theta})$ must also have mass. This forces $q$ to cover all the regions where $p$ is nonzero, typically resulting in an approximation that is too spread out but covers all modes.

In [ ]:
# Demonstrate KL directionality with a bimodal target
x_grid = np.linspace(-6, 8, 1000)

# Bimodal target: mixture of two Gaussians
target = 0.4 * stats.norm.pdf(x_grid, -1, 0.8) + 0.6 * stats.norm.pdf(x_grid, 3, 1.0)

# Reverse KL minimizer (mode-seeking): single Gaussian near the dominant mode
q_reverse = stats.norm.pdf(x_grid, 3.0, 0.95)

# Forward KL minimizer (mass-covering): wide Gaussian covering both modes
q_forward = stats.norm.pdf(x_grid, 1.5, 2.5)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(x_grid, target, "k-", linewidth=2, label="Target $p$")
axes[0].fill_between(x_grid, target, alpha=0.15, color="black")
axes[0].plot(x_grid, q_reverse, "r--", linewidth=2, label="$q$ (reverse KL)")
axes[0].set_title("Reverse KL: $\\mathrm{KL}(q \\| p)$ — Mode-Seeking")
axes[0].set_xlabel("$\\theta$")
axes[0].set_ylabel("Density")
axes[0].legend()

axes[1].plot(x_grid, target, "k-", linewidth=2, label="Target $p$")
axes[1].fill_between(x_grid, target, alpha=0.15, color="black")
axes[1].plot(x_grid, q_forward, "b--", linewidth=2, label="$q$ (forward KL)")
axes[1].set_title("Forward KL: $\\mathrm{KL}(p \\| q)$ — Mass-Covering")
axes[1].set_xlabel("$\\theta$")
axes[1].set_ylabel("Density")
axes[1].legend()

plt.tight_layout()

The reverse KL approximation captures one mode well but ignores the other entirely. The forward KL approximation covers both modes but overestimates the spread and places mass in the low-density region between the modes. Variational inference uses reverse KL, so it tends to underestimate posterior uncertainty and may miss multimodality.

### Question

Consider a bimodal posterior with modes at $\theta = -2$ and $\theta = 3$, where the mode at $\theta = 3$ has twice the probability mass. You approximate this posterior with a single Gaussian $q(\theta) = N(\mu, \sigma^2)$ by minimizing $\text{KL}(q \| p)$.

(a) Where would you expect $\mu$ to be located? Would $q$ cover both modes?

(b) If you instead minimized $\text{KL}(p \| q)$, how would the result differ?

(c) Which approximation would give more reliable credible intervals?

### Answer

(a) The reverse KL minimizer would place $\mu$ near $\theta = 3$, the dominant mode. Because $\text{KL}(q \| p)$ penalizes $q$ for placing mass where $p$ is small, the Gaussian will not try to bridge the gap between the modes. It will concentrate on the mode with higher density, resulting in $\sigma$ that roughly matches the width of that mode. The left mode near $\theta = -2$ would be effectively ignored.

(b) Minimizing $\text{KL}(p \| q)$ would produce a much wider Gaussian centered somewhere between the two modes (roughly at the mean of $p$, which is closer to $\theta = 1$). The variance would be large enough to cover both modes, but the approximation would place substantial mass in the valley between the modes where $p$ is nearly zero.

(c) Neither gives fully reliable credible intervals for this bimodal case. The reverse KL approximation gives intervals that are too narrow and miss one mode entirely. The forward KL approximation gives intervals that are too wide. For multimodal posteriors, MCMC or more expressive variational families (e.g., mixture of Gaussians) are needed. This is a fundamental limitation of unimodal variational approximations.

## The Evidence Lower Bound (ELBO)

### Derivation

We want to minimize $\text{KL}(q(\boldsymbol{\theta}) \| p(\boldsymbol{\theta} | \mathbf{y}))$, but this requires the intractable posterior $p(\boldsymbol{\theta} | \mathbf{y})$. The key insight is to decompose the log-evidence (marginal log-likelihood) in a way that avoids this.

Start with the definition of KL divergence:

$$\text{KL}(q \| p(\cdot | \mathbf{y})) = E_q\left[\log \frac{q(\boldsymbol{\theta})}{p(\boldsymbol{\theta} | \mathbf{y})}\right] = E_q[\log q(\boldsymbol{\theta})] - E_q[\log p(\boldsymbol{\theta} | \mathbf{y})]$$

Substituting $p(\boldsymbol{\theta} | \mathbf{y}) = p(\mathbf{y}, \boldsymbol{\theta}) / p(\mathbf{y})$:

$$\text{KL}(q \| p(\cdot | \mathbf{y})) = E_q[\log q(\boldsymbol{\theta})] - E_q[\log p(\mathbf{y}, \boldsymbol{\theta})] + \log p(\mathbf{y})$$

Rearranging:

$$\log p(\mathbf{y}) = \underbrace{E_q[\log p(\mathbf{y}, \boldsymbol{\theta})] - E_q[\log q(\boldsymbol{\theta})]}_{\text{ELBO}(q)} + \text{KL}(q \| p(\cdot | \mathbf{y}))$$

The left side, $\log p(\mathbf{y})$, is the log-evidence (marginal log-likelihood). It is a constant with respect to $q$. Since $\text{KL} \geq 0$:

$$\log p(\mathbf{y}) \geq \text{ELBO}(q) = E_q[\log p(\mathbf{y}, \boldsymbol{\theta})] - E_q[\log q(\boldsymbol{\theta})]$$

This is the **evidence lower bound (ELBO)**. Because $\log p(\mathbf{y})$ is fixed, maximizing the ELBO is equivalent to minimizing $\text{KL}(q \| p(\cdot | \mathbf{y}))$. We have converted an intractable KL minimization into a tractable optimization problem: the ELBO depends only on the joint distribution $p(\mathbf{y}, \boldsymbol{\theta})$ (which we can evaluate) and the variational distribution $q(\boldsymbol{\theta})$ (which we control).

### Alternative Derivation via Jensen's Inequality

There is a more direct derivation using Jensen's inequality. Start from the log-evidence and introduce $q$:

$$\log p(\mathbf{y}) = \log \int p(\mathbf{y}, \boldsymbol{\theta}) \, d\boldsymbol{\theta} = \log \int \frac{p(\mathbf{y}, \boldsymbol{\theta})}{q(\boldsymbol{\theta})} q(\boldsymbol{\theta}) \, d\boldsymbol{\theta} = \log E_q\left[\frac{p(\mathbf{y}, \boldsymbol{\theta})}{q(\boldsymbol{\theta})}\right]$$

By Jensen's inequality ($\log$ is concave):

$$\log E_q\left[\frac{p(\mathbf{y}, \boldsymbol{\theta})}{q(\boldsymbol{\theta})}\right] \geq E_q\left[\log \frac{p(\mathbf{y}, \boldsymbol{\theta})}{q(\boldsymbol{\theta})}\right] = \text{ELBO}(q)$$

This gives the same lower bound. The gap between $\log p(\mathbf{y})$ and the ELBO is exactly $\text{KL}(q \| p(\cdot | \mathbf{y}))$.

### Decomposition of the ELBO

The ELBO can be rewritten in a form that is easier to interpret:

$$\text{ELBO}(q) = E_q[\log p(\mathbf{y}, \boldsymbol{\theta})] - E_q[\log q(\boldsymbol{\theta})]$$

$$= E_q[\log p(\mathbf{y} | \boldsymbol{\theta})] + E_q[\log p(\boldsymbol{\theta})] - E_q[\log q(\boldsymbol{\theta})]$$

$$= \underbrace{E_q[\log p(\mathbf{y} | \boldsymbol{\theta})]}_{\text{expected log-likelihood}} - \underbrace{\text{KL}(q(\boldsymbol{\theta}) \| p(\boldsymbol{\theta}))}_{\text{complexity penalty}}$$

The first term rewards $q$ for concentrating on parameter values that explain the data well. The second term penalizes $q$ for deviating from the prior. This mirrors the structure of regularized maximum likelihood: the expected log-likelihood acts as a data-fit term, and the KL divergence to the prior acts as a regularizer. At the optimum, $q$ balances fitting the data against staying close to the prior.

### Connection to EM

The EM algorithm can be viewed as a special case of variational inference. In EM, the E-step computes the posterior $p(\mathbf{Z} | \mathbf{Y}, \boldsymbol{\theta}^{(t)})$ over latent variables given the current parameters, and the M-step maximizes the expected complete-data log-likelihood. If we write the ELBO with the variational distribution $q(\mathbf{Z})$ for the latent variables and treat $\boldsymbol{\theta}$ as a fixed parameter to optimize:

- The **E-step** sets $q(\mathbf{Z}) = p(\mathbf{Z} | \mathbf{Y}, \boldsymbol{\theta}^{(t)})$, which maximizes the ELBO with respect to $q$ and makes the KL gap zero.
- The **M-step** maximizes the ELBO with respect to $\boldsymbol{\theta}$, which is exactly maximizing $Q(\boldsymbol{\theta} | \boldsymbol{\theta}^{(t)})$.

In full variational inference, we approximate the posterior over *all* unknowns (both parameters and latent variables) with a tractable $q$, rather than computing the exact conditional as EM does. This makes VI more general but introduces approximation error.

### Question

The ELBO satisfies $\log p(\mathbf{y}) = \text{ELBO}(q) + \text{KL}(q \| p(\cdot | \mathbf{y}))$.

(a) If we could optimize over all possible distributions $q$ without any restrictions, what would $q^*$ be? What would the ELBO equal?

(b) In practice, we restrict $q$ to a tractable family $\mathcal{Q}$. How does this restriction affect the gap between the ELBO and $\log p(\mathbf{y})$?

(c) The ELBO decomposes into an expected log-likelihood term and a KL penalty term. What happens to the approximation $q$ if we drop the KL penalty and only maximize $E_q[\log p(\mathbf{y} | \boldsymbol{\theta})]$?

### Answer

(a) Without restrictions, the optimal $q^*$ is the true posterior $p(\boldsymbol{\theta} | \mathbf{y})$ itself. At this point, $\text{KL}(q^* \| p(\cdot | \mathbf{y})) = 0$ and $\text{ELBO}(q^*) = \log p(\mathbf{y})$. The bound is tight.

(b) Restricting $q$ to a family $\mathcal{Q}$ means the best we can do is find the closest member of $\mathcal{Q}$ to the true posterior. The residual $\text{KL}(q^* \| p(\cdot | \mathbf{y})) > 0$ (unless the true posterior happens to lie in $\mathcal{Q}$), and the ELBO is strictly less than $\log p(\mathbf{y})$. The gap equals this residual KL, which measures how well the family $\mathcal{Q}$ can represent the posterior.

(c) If we drop the KL penalty, the optimization would push $q$ to concentrate all its mass on the parameter value $\hat{\boldsymbol{\theta}}$ that maximizes the likelihood. This gives the maximum likelihood estimate (a point mass), which ignores prior information and posterior uncertainty. The KL penalty is essential for $q$ to reflect genuine posterior uncertainty rather than collapsing to a point estimate.

## Variational Family Expressiveness: 2D Gaussian Mixture

To illustrate how the choice of variational family affects approximation quality, we approximate a 2D Gaussian mixture with 3 components using a single bivariate normal under three variational families:

1. **Isotropic**: $q(\mathbf{x}) = N(\boldsymbol{\mu}, \sigma^2 \mathbf{I})$ — 3 parameters
2. **Diagonal**: $q(\mathbf{x}) = N(\boldsymbol{\mu}, \mathrm{diag}(\sigma_1^2, \sigma_2^2))$ — 4 parameters
3. **Full covariance**: $q(\mathbf{x}) = N(\boldsymbol{\mu}, \boldsymbol{\Sigma})$ — 5 parameters

We minimize $\mathrm{KL}(q \| p)$ for each family using the reparameterization trick with fixed Monte Carlo samples. With fixed samples $\boldsymbol{\epsilon}_1, \ldots, \boldsymbol{\epsilon}_S \sim N(\mathbf{0}, \mathbf{I})$, the objective becomes a deterministic function of the variational parameters that can be optimized with standard methods.

In [ ]:
# 2D Gaussian mixture target with 3 components
mix_weights = np.array([0.4, 0.35, 0.25])
mix_means = np.array([[-2.5, -0.5], [0.0, 0.5], [2.5, 2.0]])
mix_covs = np.array([
    [[0.8, 0.3], [0.3, 0.5]],
    [[0.6, 0.2], [0.2, 0.4]],
    [[0.7, -0.1], [-0.1, 0.5]],
])


def log_gmm_2d(x):
    """Log density of the 2D Gaussian mixture."""
    log_comps = np.column_stack([
        np.log(mix_weights[k])
        + stats.multivariate_normal.logpdf(x, mix_means[k], mix_covs[k])
        for k in range(3)
    ])
    mx = log_comps.max(axis=1, keepdims=True)
    return mx.ravel() + np.log(np.exp(log_comps - mx).sum(axis=1))


def fit_gaussian_vi(family, n_mc=10000, n_restarts=10, rng=None):
    """Minimize KL(q || p) for a bivariate Gaussian q over the GMM target.

    Parameters
    ----------
    family : str, one of 'isotropic', 'diagonal', 'full'
    n_mc : int, number of fixed MC samples for KL estimation
    n_restarts : int, number of random restarts
    rng : numpy Generator for random initialization (used only for
        starting points; MC samples use a fixed seed for fair comparison)

    Returns
    -------
    dict with mu, cov, kl (estimated KL divergence)
    """
    if rng is None:
        rng = np.random.default_rng()
    # Fixed MC samples (common random numbers) so KL values are comparable
    eps = np.random.default_rng(0).normal(0, 1, (n_mc, 2))

    def kl_objective(params):
        """KL(q || p) = -H(q) - E_q[log p(x)], estimated via MC."""
        mu = params[:2]
        if family == "isotropic":
            ls = params[2]
            x = mu + np.exp(ls) * eps
            ent = 1 + np.log(2 * np.pi) + 2 * ls
        elif family == "diagonal":
            ls = params[2:4]
            x = mu + np.exp(ls) * eps
            ent = 1 + np.log(2 * np.pi) + ls.sum()
        else:  # full
            lL11, L21, lL22 = params[2], params[3], params[4]
            L = np.array([[np.exp(lL11), 0.0], [L21, np.exp(lL22)]])
            x = mu + eps @ L.T
            ent = 1 + np.log(2 * np.pi) + lL11 + lL22
        return -ent - np.mean(log_gmm_2d(x))

    n_params = {"isotropic": 3, "diagonal": 4, "full": 5}[family]
    best = None
    for _ in range(n_restarts):
        x0 = np.concatenate(
            [rng.uniform(-3, 3, 2), rng.normal(0, 0.5, n_params - 2)]
        )
        res = minimize(
            kl_objective,
            x0,
            method="Nelder-Mead",
            options={"maxiter": 20000, "xatol": 1e-10, "fatol": 1e-10},
        )
        if best is None or res.fun < best.fun:
            best = res

    params = best.x
    mu = params[:2]
    if family == "isotropic":
        s = np.exp(params[2])
        cov = s**2 * np.eye(2)
    elif family == "diagonal":
        s = np.exp(params[2:4])
        cov = np.diag(s**2)
    else:
        L = np.array(
            [[np.exp(params[2]), 0.0], [params[3], np.exp(params[4])]]
        )
        cov = L @ L.T

    return {"mu": mu, "cov": cov, "kl": best.fun}

In [ ]:
# Fit each variational family
rng_2d = np.random.default_rng(42)
results_2d = {}
for fam in ["isotropic", "diagonal", "full"]:
    results_2d[fam] = fit_gaussian_vi(fam, rng=rng_2d)
    r = results_2d[fam]
    print(
        f"{fam:12s}: mu=[{r['mu'][0]:+.3f}, {r['mu'][1]:+.3f}], "
        f"KL={r['kl']:.4f}"
    )
    print(
        f"{'':12s}  cov=[[{r['cov'][0,0]:.3f}, {r['cov'][0,1]:.3f}], "
        f"[{r['cov'][1,0]:.3f}, {r['cov'][1,1]:.3f}]]"
    )

In [ ]:
# Evaluate densities on a grid
lim = 6
g = np.linspace(-lim, lim, 200)
G1, G2 = np.meshgrid(g, g)
grid_pts = np.column_stack([G1.ravel(), G2.ravel()])
P_target = np.exp(log_gmm_2d(grid_pts)).reshape(G1.shape)

fig, axes = plt.subplots(2, 2, figsize=(11, 10))

# Target density
ax = axes[0, 0]
ax.contourf(G1, G2, P_target, levels=20, cmap="Greys")
for k in range(3):
    ax.plot(*mix_means[k], "r+", markersize=12, markeredgewidth=2)
ax.set_title("Target: 3-Component GMM")
ax.set_xlabel("$x_1$")
ax.set_ylabel("$x_2$")
ax.set_aspect("equal")

# Each approximation overlaid on target contours
families = ["isotropic", "diagonal", "full"]
labels = [
    "Isotropic: $q = N(\\mu, \\sigma^2 I)$",
    "Diagonal: $q = N(\\mu, \\mathrm{diag}(\\sigma_1^2, \\sigma_2^2))$",
    "Full: $q = N(\\mu, \\Sigma)$",
]
line_colors = ["tab:red", "tab:blue", "tab:green"]
fill_cmaps = ["Reds", "Blues", "Greens"]

for i, (fam, label, lc, cm) in enumerate(
    zip(families, labels, line_colors, fill_cmaps)
):
    ax = axes[(i + 1) // 2, (i + 1) % 2]
    r = results_2d[fam]
    Q = stats.multivariate_normal.pdf(
        grid_pts, r["mu"], r["cov"]
    ).reshape(G1.shape)
    ax.contour(
        G1, G2, P_target, levels=8, colors="gray", alpha=0.5, linewidths=0.8
    )
    ax.contourf(G1, G2, Q, levels=15, cmap=cm, alpha=0.5)
    ax.contour(G1, G2, Q, levels=6, colors=lc, linewidths=1.5)
    ax.plot(*r["mu"], "+", color="black", markersize=12, markeredgewidth=2)
    for k in range(3):
        ax.plot(
            *mix_means[k], "x", color="gray", markersize=8, markeredgewidth=1.5
        )
    ax.set_title(f"{label}\nKL$(q \\| p)$ = {r['kl']:.3f}")
    ax.set_xlabel("$x_1$")
    ax.set_ylabel("$x_2$")
    ax.set_aspect("equal")

plt.tight_layout()

The three approximations illustrate how variational family expressiveness affects the quality of the VI approximation:

- **Isotropic** and **diagonal** both collapse onto the dominant mode due to the mode-seeking behavior of the reverse KL. Because the local shape of a single mode is roughly circular, the diagonal family gains little over the isotropic family (KL improvement is small).
- **Full covariance** finds a qualitatively different solution: it tilts an ellipse along the diagonal axis connecting the three modes, covering far more of the target density and achieving a much lower KL divergence.

In this example, the covariance *structure* (off-diagonal entries) matters more than simply allowing unequal variances, because the target's mass is distributed along a tilted axis that axis-aligned ellipses cannot efficiently capture. More broadly, no single Gaussian, regardless of its covariance parameterization, can represent multimodality. To capture all three modes faithfully, one would need a more expressive variational family such as a mixture of Gaussians or a normalizing flow.

## Mean-Field Variational Inference

### The Mean-Field Assumption

The most common choice for the variational family $\mathcal{Q}$ is the **mean-field** family, which assumes that the latent variables are independent under $q$:

$$q(\boldsymbol{\theta}) = \prod_{j=1}^{d} q_j(\theta_j)$$

Each factor $q_j$ has its own parameters (called **variational parameters**) that are optimized to maximize the ELBO. The mean-field assumption ignores all posterior correlations between parameters. This is a strong simplification, but it makes the optimization tractable and each factor can often be computed in closed form.

### Optimal Mean-Field Factors

A remarkable result from variational calculus gives the optimal form of each factor. Holding all other factors $q_{-j} = \{q_k : k \neq j\}$ fixed, the ELBO-maximizing $q_j$ is:

$$\log q_j^*(\theta_j) = E_{q_{-j}}[\log p(\mathbf{y}, \boldsymbol{\theta})] + \text{const}$$

That is, $q_j^*(\theta_j) \propto \exp\left(E_{q_{-j}}[\log p(\mathbf{y}, \boldsymbol{\theta})]\right)$. The optimal variational factor is the exponentiated expected log of the joint distribution, where the expectation is over all other factors. For models in the exponential family with conjugate priors, this yields closed-form updates because the expected log-joint is a linear function of the sufficient statistics.

### Coordinate Ascent Variational Inference (CAVI)

The **CAVI** algorithm optimizes the ELBO by cyclically updating each variational factor while holding the others fixed. Each update is guaranteed to increase (or leave unchanged) the ELBO, analogous to how each EM step increases the observed-data log-likelihood.

```
CAVI Algorithm:
1. Initialize variational parameters for each factor q_j
2. While ELBO has not converged:
   For j = 1, ..., d:
     Update q_j(θ_j) ∝ exp( E_{q_{-j}}[log p(y, θ)] )
   Compute ELBO
3. Return q(θ) = ∏_j q_j(θ_j)
```

CAVI is the variational analog of Gibbs sampling: Gibbs samples each parameter from its full conditional, while CAVI sets each variational factor to the exponentiated expectation of the log full conditional. Both cycle through the parameters one at a time, and both converge under mild conditions.

| Property | CAVI | Gibbs Sampling |
|----------|------|----------------|
| Updates each component | Optimization (closed form) | Sampling from conditional |
| Output | Approximate distribution $q$ | Samples from exact posterior |
| Monotone convergence | ELBO increases | Log-posterior (stationary dist.) |
| Speed per iteration | Fast (analytic updates) | One sample per iteration |
| Accuracy | Approximate | Asymptotically exact |
| Correlation handling | Ignored (mean-field) | Captured (but slow mixing) |

## Example: Gaussian with Unknown Mean and Precision

To build intuition for CAVI, we start with a simple conjugate model where the exact posterior is available for comparison.

### Model

Suppose we observe $y_1, \ldots, y_n$ from a normal distribution with unknown mean $\mu$ and unknown precision $\tau = 1/\sigma^2$:

$$y_i | \mu, \tau \sim N(\mu, \tau^{-1})$$

With conjugate priors:

$$\mu \sim N(\mu_0, (\kappa_0 \tau)^{-1}), \qquad \tau \sim \text{Gamma}(a_0, b_0)$$

This is the normal-gamma conjugate model. The exact posterior is a normal-gamma distribution $p(\mu, \tau | \mathbf{y}) = p(\mu | \tau, \mathbf{y}) \, p(\tau | \mathbf{y})$, which we can compute in closed form for validation.

### Mean-Field Approximation

We approximate $p(\mu, \tau | \mathbf{y})$ with the mean-field family $q(\mu, \tau) = q_\mu(\mu) \, q_\tau(\tau)$. This assumes $\mu$ and $\tau$ are independent under $q$, which is an approximation since the true posterior couples them.

Applying the optimal mean-field formula, we find:

- $q_\mu(\mu) = N(\mu_n, \sigma_n^2)$ where:
  - $\sigma_n^2 = (\kappa_0 E_{q_\tau}[\tau] + n E_{q_\tau}[\tau])^{-1} = ((\kappa_0 + n) E_{q_\tau}[\tau])^{-1}$
  - $\mu_n = \sigma_n^2 (\kappa_0 E_{q_\tau}[\tau] \mu_0 + E_{q_\tau}[\tau] \sum_i y_i)$

- $q_\tau(\tau) = \text{Gamma}(a_n, b_n)$ where:
  - $a_n = a_0 + n/2$
  - $b_n = b_0 + \frac{1}{2} \sum_i (y_i^2 - 2 y_i E_{q_\mu}[\mu] + E_{q_\mu}[\mu^2]) + \frac{\kappa_0}{2}(E_{q_\mu}[\mu^2] - 2 \mu_0 E_{q_\mu}[\mu] + \mu_0^2)$

These updates depend on moments from the other factor: $q_\mu$ needs $E_{q_\tau}[\tau] = a_n / b_n$, and $q_\tau$ needs $E_{q_\mu}[\mu] = \mu_n$ and $E_{q_\mu}[\mu^2] = \sigma_n^2 + \mu_n^2$.

### Implementation

In [ ]:
def cavi_normal_gamma(y, mu0=0.0, kappa0=0.01, a0=0.01, b0=0.01,
                      max_iter=200, tol=1e-8):
    """CAVI for Normal-Gamma model: y_i ~ N(mu, 1/tau).

    Priors: mu ~ N(mu0, 1/(kappa0*tau)), tau ~ Gamma(a0, b0).
    Variational family: q(mu, tau) = q_mu(mu) * q_tau(tau).

    Returns
    -------
    dict with variational parameters and ELBO history.
    """
    from scipy.special import digamma, gammaln
    n = len(y)
    sum_y = np.sum(y)
    sum_y2 = np.sum(y**2)

    # Initialize
    a_n = a0 + n / 2.0
    b_n = b0 + 0.5 * np.var(y) * n
    E_tau = a_n / b_n

    elbo_history = []

    for iteration in range(max_iter):
        # Update q_mu: N(mu_n, sigma_n^2)
        sigma_n2 = 1.0 / ((kappa0 + n) * E_tau)
        mu_n = sigma_n2 * (kappa0 * E_tau * mu0 + E_tau * sum_y)
        E_mu = mu_n
        E_mu2 = sigma_n2 + mu_n**2

        # Update q_tau: Gamma(a_n, b_n)
        a_n = a0 + n / 2.0
        b_n = (b0
               + 0.5 * (sum_y2 - 2 * sum_y * E_mu + n * E_mu2)
               + 0.5 * kappa0 * (E_mu2 - 2 * mu0 * E_mu + mu0**2))
        E_tau = a_n / b_n
        E_log_tau = digamma(a_n) - np.log(b_n)

        # Compute ELBO
        # E_q[log p(y | mu, tau)]
        elbo_lik = (0.5 * n * (E_log_tau - np.log(2 * np.pi))
                    - 0.5 * E_tau * (sum_y2 - 2 * sum_y * E_mu + n * E_mu2))

        # E_q[log p(mu | tau)]
        elbo_prior_mu = (0.5 * (np.log(kappa0) + E_log_tau - np.log(2 * np.pi))
                         - 0.5 * kappa0 * E_tau * (E_mu2 - 2 * mu0 * E_mu + mu0**2))

        # E_q[log p(tau)]
        elbo_prior_tau = ((a0 - 1) * E_log_tau - b0 * E_tau
                          + a0 * np.log(b0) - gammaln(a0))

        # -E_q[log q_mu]  (entropy of N(mu_n, sigma_n^2))
        entropy_mu = 0.5 * np.log(2 * np.pi * np.e * sigma_n2)

        # -E_q[log q_tau]  (entropy of Gamma(a_n, b_n))
        entropy_tau = (a_n - np.log(b_n) + gammaln(a_n)
                       + (1 - a_n) * digamma(a_n))

        elbo = elbo_lik + elbo_prior_mu + elbo_prior_tau + entropy_mu + entropy_tau
        elbo_history.append(elbo)

        if iteration > 0 and abs(elbo_history[-1] - elbo_history[-2]) < tol:
            break

    return {
        "mu_n": mu_n, "sigma_n2": sigma_n2,
        "a_n": a_n, "b_n": b_n,
        "elbo_history": elbo_history,
    }

In [ ]:
# Simulate data
np.random.seed(42)
n_data = 100
mu_true = 3.0
tau_true = 2.0  # sigma^2 = 0.5
y_data = np.random.normal(mu_true, 1.0 / np.sqrt(tau_true), n_data)

# Run CAVI
result_cavi = cavi_normal_gamma(y_data, mu0=0.0, kappa0=0.01, a0=0.01, b0=0.01)

print("Variational approximation:")
print(f"  q(mu) = N({result_cavi['mu_n']:.4f}, {result_cavi['sigma_n2']:.6f})")
print(f"  q(tau) = Gamma({result_cavi['a_n']:.2f}, {result_cavi['b_n']:.4f})")
print(f"  E[mu] = {result_cavi['mu_n']:.4f} (true: {mu_true})")
print(f"  E[tau] = {result_cavi['a_n'] / result_cavi['b_n']:.4f} (true: {tau_true})")
print(f"  Iterations: {len(result_cavi['elbo_history'])}")

### Comparison with Exact Posterior

Since this model is conjugate, we can compute the exact posterior and compare.

In [ ]:
# Exact posterior parameters (normal-gamma conjugate update)
kappa0, mu0, a0, b0 = 0.01, 0.0, 0.01, 0.01
y_bar = np.mean(y_data)
n_exact = len(y_data)
kappa_n = kappa0 + n_exact
mu_n_exact = (kappa0 * mu0 + n_exact * y_bar) / kappa_n
a_n_exact = a0 + n_exact / 2.0
b_n_exact = (b0
             + 0.5 * np.sum((y_data - y_bar)**2)
             + 0.5 * kappa0 * n_exact * (y_bar - mu0)**2 / kappa_n)

# The exact marginal posterior of mu is a t-distribution
# The exact marginal posterior of tau is Gamma(a_n_exact, b_n_exact)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# ELBO convergence
axes[0].plot(result_cavi["elbo_history"], "o-", markersize=3, color="steelblue")
axes[0].set_xlabel("Iteration")
axes[0].set_ylabel("ELBO")
axes[0].set_title("CAVI Convergence")

# q(mu) vs exact marginal
mu_grid = np.linspace(2.5, 3.5, 300)
q_mu_pdf = stats.norm.pdf(mu_grid, result_cavi["mu_n"],
                           np.sqrt(result_cavi["sigma_n2"]))

# Exact marginal of mu is t-distributed
exact_mu_scale = np.sqrt(b_n_exact / (a_n_exact * kappa_n))
exact_mu_pdf = stats.t.pdf(mu_grid, 2 * a_n_exact,
                            loc=mu_n_exact, scale=exact_mu_scale)

axes[1].plot(mu_grid, q_mu_pdf, "r-", linewidth=2, label="VI: $q(\\mu)$")
axes[1].plot(mu_grid, exact_mu_pdf, "k--", linewidth=2, label="Exact marginal")
axes[1].axvline(mu_true, color="gray", linestyle=":", label="True $\\mu$")
axes[1].set_xlabel("$\\mu$")
axes[1].set_ylabel("Density")
axes[1].set_title("Posterior of $\\mu$")
axes[1].legend()

# q(tau) vs exact marginal
tau_grid = np.linspace(0.5, 4, 300)
q_tau_pdf = stats.gamma.pdf(tau_grid, result_cavi["a_n"],
                             scale=1.0 / result_cavi["b_n"])
exact_tau_pdf = stats.gamma.pdf(tau_grid, a_n_exact,
                                 scale=1.0 / b_n_exact)

axes[2].plot(tau_grid, q_tau_pdf, "r-", linewidth=2, label="VI: $q(\\tau)$")
axes[2].plot(tau_grid, exact_tau_pdf, "k--", linewidth=2, label="Exact marginal")
axes[2].axvline(tau_true, color="gray", linestyle=":", label="True $\\tau$")
axes[2].set_xlabel("$\\tau$")
axes[2].set_ylabel("Density")
axes[2].set_title("Posterior of $\\tau$")
axes[2].legend()

plt.tight_layout()

The VI marginal distributions closely match the exact marginals in this conjugate case. The mean-field assumption forces the approximation to be a product $q_\mu \cdot q_\tau$, while the true posterior has some correlation between $\mu$ and $\tau$. The marginal means and variances can still be accurate even though the joint distribution is misrepresented.

### Question

In the CAVI updates, $q_\mu$ depends on $E_{q_\tau}[\tau]$ and $q_\tau$ depends on $E_{q_\mu}[\mu]$ and $E_{q_\mu}[\mu^2]$.

(a) How is this circular dependency resolved?

(b) Is CAVI guaranteed to converge to the global optimum of the ELBO? Why or why not?

(c) In what sense is CAVI analogous to the EM algorithm?

### Answer

(a) The circular dependency is resolved by iteration: we initialize one set of variational parameters, use them to update the other, and alternate. Each update uses the *current* expectations from the other factor. This is exactly the coordinate ascent strategy: optimize one coordinate while holding the others fixed, then move to the next coordinate.

(b) CAVI is guaranteed to converge to a *local* optimum of the ELBO, not necessarily the global optimum. Each coordinate update increases the ELBO (or leaves it unchanged), so the sequence of ELBO values is non-decreasing and bounded above. However, the ELBO may be non-convex, so different initializations can lead to different local optima, just like with EM.

(c) CAVI is analogous to EM in several ways: both alternate between two types of updates, both guarantee monotone improvement of an objective function (ELBO for CAVI, observed-data log-likelihood for EM), both converge to a stationary point, and both are sensitive to initialization. The E-step of EM can be seen as a variational update where $q(\mathbf{Z})$ is set to the exact conditional (without any mean-field restriction), while the M-step optimizes the parameters. CAVI applies the same alternating optimization but with a restricted variational family.

## Example: Gaussian Mixture Model

The Gaussian mixture model (GMM) is the canonical example for variational inference, paralleling our EM treatment from earlier in the course.

### Model Setup

We observe $\mathbf{x} = (x_1, \ldots, x_n)$ from a $K$-component mixture:

$$x_i | c_i = k \sim N(\mu_k, 1), \qquad c_i \sim \text{Categorical}(\pi_1, \ldots, \pi_K)$$

$$\mu_k \sim N(0, \sigma_0^2) \quad \text{for } k = 1, \ldots, K$$

where $c_i \in \{1, \ldots, K\}$ is the cluster assignment for observation $i$ and $\sigma_0^2$ is a fixed prior variance. For simplicity, we fix the mixing proportions $\pi_k = 1/K$ and the component variances to 1.

### Mean-Field Variational Family

We approximate the joint posterior $p(\boldsymbol{\mu}, \mathbf{c} | \mathbf{x})$ with:

$$q(\boldsymbol{\mu}, \mathbf{c}) = \prod_{k=1}^K q_{\mu_k}(\mu_k) \prod_{i=1}^n q_{c_i}(c_i)$$

where $q_{\mu_k}(\mu_k) = N(m_k, s_k^2)$ and $q_{c_i}(c_i) = \text{Categorical}(\phi_{i1}, \ldots, \phi_{iK})$.

The variational parameters are $\{m_k, s_k^2\}_{k=1}^K$ (means and variances of the cluster center approximations) and $\{\phi_{ik}\}_{i,k}$ (soft assignment probabilities).

### CAVI Updates

The optimal updates are:

**Assignment update:** For each observation $i$ and component $k$:

$$\phi_{ik} \propto \exp\left(E_{q_{\mu_k}}[\mu_k] \cdot x_i - \frac{1}{2} E_{q_{\mu_k}}[\mu_k^2]\right) = \exp\left(m_k x_i - \frac{1}{2}(s_k^2 + m_k^2)\right)$$

then normalize so $\sum_k \phi_{ik} = 1$.

**Mean update:** For each component $k$:

$$s_k^2 = \frac{1}{1/\sigma_0^2 + \sum_i \phi_{ik}}, \qquad m_k = s_k^2 \sum_i \phi_{ik} \, x_i$$

These updates closely mirror the EM updates from the EM lecture: the assignment update is analogous to the E-step (computing responsibilities), and the mean update is analogous to the M-step (computing weighted means). The key difference is that VI maintains uncertainty about the cluster means ($s_k^2 > 0$) while EM uses point estimates.

### Implementation

In [ ]:
def cavi_gmm(x, K, sigma0=5.0, max_iter=200, tol=1e-8, rng=None):
    """CAVI for Gaussian mixture model with known variance.

    Model: x_i | c_i=k ~ N(mu_k, 1), mu_k ~ N(0, sigma0^2).
    Mixing proportions fixed at 1/K.

    Parameters
    ----------
    x : array of shape (n,)
    K : int, number of components
    sigma0 : float, prior standard deviation for mu_k
    max_iter : int
    tol : float

    Returns
    -------
    dict with m, s2, phi, elbo_history
    """
    if rng is None:
        rng = np.random.default_rng()
    n = len(x)

    # Initialize variational parameters
    m = rng.normal(np.mean(x), np.std(x), K)
    s2 = np.ones(K)
    phi = np.ones((n, K)) / K

    elbo_history = []

    for iteration in range(max_iter):
        # Update assignments phi
        for k in range(K):
            phi[:, k] = m[k] * x - 0.5 * (s2[k] + m[k]**2)
        # Normalize in log space for stability
        phi -= phi.max(axis=1, keepdims=True)
        phi = np.exp(phi)
        phi /= phi.sum(axis=1, keepdims=True)

        # Update component means
        for k in range(K):
            n_k = np.sum(phi[:, k])
            s2[k] = 1.0 / (1.0 / sigma0**2 + n_k)
            m[k] = s2[k] * np.sum(phi[:, k] * x)

        # Compute ELBO
        E_mu = m
        E_mu2 = s2 + m**2

        # E[log p(x | c, mu)]
        elbo = 0.0
        for k in range(K):
            elbo += np.sum(phi[:, k] * (
                -0.5 * np.log(2 * np.pi)
                - 0.5 * (x**2 - 2 * x * E_mu[k] + E_mu2[k])
            ))

        # E[log p(c)] - E[log q(c)]
        for k in range(K):
            elbo += np.sum(phi[:, k] * (-np.log(K)))  # uniform prior
            safe_phi = np.clip(phi[:, k], 1e-15, 1.0)
            elbo -= np.sum(phi[:, k] * np.log(safe_phi))

        # E[log p(mu)] - E[log q(mu)]
        for k in range(K):
            elbo += -0.5 * np.log(2 * np.pi * sigma0**2) - 0.5 * E_mu2[k] / sigma0**2
            elbo += 0.5 * np.log(2 * np.pi * np.e * s2[k])

        elbo_history.append(elbo)
        if iteration > 0 and abs(elbo_history[-1] - elbo_history[-2]) < tol:
            break

    return {"m": m, "s2": s2, "phi": phi, "elbo_history": elbo_history}

### Comparison with EM

In [ ]:
# Simulate mixture data
np.random.seed(42)
n_mix = 300
pi_true_mix = np.array([0.35, 0.65])
mu_true_mix = np.array([-2.0, 3.0])
c_true = np.random.choice(2, size=n_mix, p=pi_true_mix)
x_mix = np.random.normal(mu_true_mix[c_true], 1.0)

# Run CAVI
rng_vi = np.random.default_rng(42)
vi_result = cavi_gmm(x_mix, K=2, sigma0=5.0, max_iter=200, rng=rng_vi)

# Run EM for comparison (from the EM lecture)
from scipy import stats as sp_stats

def em_gmm_1d(x, K, max_iter=200, tol=1e-8):
    """EM for 1D Gaussian mixture with unit variance."""
    n = len(x)
    # Initialize
    mu = np.array([np.percentile(x, 30), np.percentile(x, 70)])
    pi = np.ones(K) / K
    ll_history = []

    for iteration in range(max_iter):
        # E-step
        resp = np.zeros((n, K))
        for k in range(K):
            resp[:, k] = pi[k] * sp_stats.norm.pdf(x, mu[k], 1.0)
        resp /= resp.sum(axis=1, keepdims=True)

        # Log-likelihood
        ll = np.sum(np.log(np.sum(
            [pi[k] * sp_stats.norm.pdf(x, mu[k], 1.0) for k in range(K)],
            axis=0
        )))
        ll_history.append(ll)
        if iteration > 0 and abs(ll_history[-1] - ll_history[-2]) < tol:
            break

        # M-step
        for k in range(K):
            n_k = np.sum(resp[:, k])
            pi[k] = n_k / n
            mu[k] = np.sum(resp[:, k] * x) / n_k

    return {"mu": mu, "pi": pi, "resp": resp, "ll_history": ll_history}


em_result = em_gmm_1d(x_mix, K=2)

# Sort components by mean for consistent comparison
vi_order = np.argsort(vi_result["m"])
em_order = np.argsort(em_result["mu"])

print("Component means:")
print(f"  True:  {mu_true_mix}")
print(f"  VI:    {vi_result['m'][vi_order]}")
print(f"  EM:    {em_result['mu'][em_order]}")
print(f"\nVI uncertainty (posterior std of mu_k): {np.sqrt(vi_result['s2'][vi_order])}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# ELBO convergence
axes[0].plot(vi_result["elbo_history"], "o-", markersize=3, color="steelblue")
axes[0].set_xlabel("Iteration")
axes[0].set_ylabel("ELBO")
axes[0].set_title("CAVI Convergence (GMM)")

# Data with soft assignments
vi_assignments = vi_result["phi"][:, vi_order[1]]  # probability of component 2
scatter = axes[1].scatter(x_mix, np.zeros_like(x_mix), c=vi_assignments,
                          cmap="RdYlBu", s=20, alpha=0.7, edgecolors="none")
plt.colorbar(scatter, ax=axes[1], label="$\\phi_{i2}$ (prob. of component 2)")
axes[1].set_xlabel("$x$")
axes[1].set_yticks([])
axes[1].set_title("VI Soft Assignments")

# Posterior of cluster means
x_grid_mix = np.linspace(-5, 6, 500)
for k_idx, k in enumerate(vi_order):
    vi_pdf = sp_stats.norm.pdf(x_grid_mix, vi_result["m"][k],
                                np.sqrt(vi_result["s2"][k]))
    axes[2].plot(x_grid_mix, vi_pdf, linewidth=2,
                 label=f"$q(\\mu_{k_idx+1})$")
    axes[2].axvline(mu_true_mix[k_idx], color=f"C{k_idx}", linestyle="--",
                    alpha=0.5)
    axes[2].axvline(em_result["mu"][em_order[k_idx]], color=f"C{k_idx}",
                    linestyle=":", alpha=0.5)

axes[2].set_xlabel("$\\mu$")
axes[2].set_ylabel("Density")
axes[2].set_title("Posterior of Cluster Means")
axes[2].legend()

plt.tight_layout()

Unlike EM, which produces only point estimates of $\mu_1, \mu_2$, VI gives full posterior distributions $q(\mu_k) = N(m_k, s_k^2)$. The posterior standard deviations quantify our uncertainty about the cluster locations. For large $n$, the EM point estimates and VI posterior means converge, but VI additionally provides uncertainty quantification.

### Question

In the CAVI update for the GMM, the assignment probability is $\phi_{ik} \propto \exp(m_k x_i - \frac{1}{2}(s_k^2 + m_k^2))$.

(a) Compare this to the EM responsibility $\gamma_{ik} \propto \pi_k \phi(x_i | \mu_k, 1)$. What is the role of the $s_k^2$ term in the VI update that has no analog in EM?

(b) If we set $s_k^2 = 0$ in the VI update, what does the assignment formula reduce to?

### Answer

(a) The EM responsibility uses the point estimate $\mu_k$ to compute the Gaussian density $\phi(x_i | \mu_k, 1)$. The VI assignment uses the posterior mean $m_k$ but also accounts for uncertainty through $s_k^2$. The term $E[\mu_k^2] = s_k^2 + m_k^2$ averages over the uncertainty in $\mu_k$: we are computing the expected log-likelihood under the posterior distribution of $\mu_k$, not just evaluating it at a point. When $s_k^2$ is large (uncertain cluster mean), all observations become more equally likely under that component, leading to softer assignments.

(b) If $s_k^2 = 0$, then $E[\mu_k^2] = m_k^2$ and the assignment becomes $\phi_{ik} \propto \exp(m_k x_i - \frac{1}{2} m_k^2) = \exp(-\frac{1}{2}(x_i - m_k)^2 + \frac{1}{2} x_i^2)$. Since $\frac{1}{2} x_i^2$ is constant across $k$, this reduces to $\phi_{ik} \propto \exp(-\frac{1}{2}(x_i - m_k)^2) = \phi(x_i | m_k, 1)$ (up to the equal mixing proportion $1/K$). This is exactly the EM responsibility formula. So EM is a special case of VI where posterior uncertainty about the parameters is zero.

## Variational Inference for Bayesian Logistic Regression

Now we apply VI to a model without conjugacy. For Bayesian logistic regression, the full conditionals are not standard distributions, so we cannot derive closed-form CAVI updates as we did for the conjugate models above. Instead, we use a Gaussian variational family and optimize the ELBO directly.

### Variational Family

We approximate the posterior $p(\boldsymbol{\beta} | \mathbf{y}, \mathbf{X})$ with an independent Gaussian:

$$q(\boldsymbol{\beta}) = \prod_{j=1}^{p} N(\beta_j | \mu_j, \sigma_j^2)$$

The variational parameters are $\boldsymbol{\mu} = (\mu_1, \ldots, \mu_p)$ and $\boldsymbol{\sigma}^2 = (\sigma_1^2, \ldots, \sigma_p^2)$.

### ELBO Computation

The ELBO is:

$$\text{ELBO} = E_q[\log p(\mathbf{y} | \boldsymbol{\beta})] - \text{KL}(q(\boldsymbol{\beta}) \| p(\boldsymbol{\beta}))$$

The KL term between two Gaussians has a closed form:

$$\text{KL}(N(\mu_j, \sigma_j^2) \| N(0, \tau^2)) = \frac{1}{2}\left(\frac{\sigma_j^2 + \mu_j^2}{\tau^2} - 1 - \log\frac{\sigma_j^2}{\tau^2}\right)$$

The expected log-likelihood $E_q[\log p(\mathbf{y} | \boldsymbol{\beta})]$ has no closed form because the logistic function is not amenable to Gaussian expectations. We approximate it using Monte Carlo samples from $q$.

### Reparameterization Trick

To compute gradients of the ELBO with respect to the variational parameters, we use the **reparameterization trick**. Instead of sampling $\boldsymbol{\beta} \sim N(\boldsymbol{\mu}, \text{diag}(\boldsymbol{\sigma}^2))$ directly, we write:

$$\boldsymbol{\beta} = \boldsymbol{\mu} + \boldsymbol{\sigma} \odot \boldsymbol{\epsilon}, \qquad \boldsymbol{\epsilon} \sim N(\mathbf{0}, \mathbf{I})$$

where $\odot$ is element-wise multiplication. This moves the randomness into a fixed distribution ($\boldsymbol{\epsilon}$ does not depend on the variational parameters), allowing us to differentiate through the sampling operation. The gradient of the ELBO with respect to $\boldsymbol{\mu}$ and $\boldsymbol{\sigma}$ can then be estimated via a Monte Carlo average.

### Implementation

In [ ]:
def vi_logistic_regression(X, y, tau=10.0, n_mc=10, learning_rate=0.01,
                           max_iter=5000, tol=1e-6, rng=None):
    """Variational inference for Bayesian logistic regression.

    Variational family: q(beta) = prod_j N(beta_j | mu_j, sigma_j^2).
    Uses reparameterization trick and Monte Carlo ELBO gradient.

    Parameters
    ----------
    X : array of shape (n, p)
    y : array of shape (n,)
    tau : float, prior standard deviation
    n_mc : int, MC samples for ELBO gradient
    learning_rate : float
    max_iter : int
    tol : float

    Returns
    -------
    dict with mu, sigma, elbo_history
    """
    if rng is None:
        rng = np.random.default_rng()
    n_obs, p_dim = X.shape

    # Variational parameters: mu and log(sigma) (log for positivity)
    mu = np.zeros(p_dim)
    log_sigma = np.zeros(p_dim) - 1.0  # start with sigma ~ 0.37

    elbo_history = []

    for iteration in range(max_iter):
        sigma = np.exp(log_sigma)

        # Reparameterized samples: beta = mu + sigma * epsilon
        epsilon = rng.normal(0, 1, (n_mc, p_dim))
        beta_samples = mu + sigma * epsilon  # (n_mc, p_dim)

        # Compute ELBO and gradients
        grad_mu = np.zeros(p_dim)
        grad_log_sigma = np.zeros(p_dim)
        elbo = 0.0

        for s in range(n_mc):
            beta_s = beta_samples[s]
            eta = X @ beta_s

            # Log-likelihood
            log_lik = np.sum(y * eta - np.logaddexp(0, eta))

            # Log-prior
            log_prior = -0.5 * np.sum(beta_s**2) / tau**2

            # Log-q
            log_q = -0.5 * np.sum(((beta_s - mu) / sigma)**2) - np.sum(log_sigma)

            # ELBO contribution
            elbo += (log_lik + log_prior - log_q) / n_mc

            # Gradient of log-likelihood + log-prior w.r.t. beta
            prob_s = 1 / (1 + np.exp(-eta))
            grad_beta = X.T @ (y - prob_s) - beta_s / tau**2

            # Chain rule: d/d_mu = d/d_beta * d_beta/d_mu = grad_beta * 1
            grad_mu += grad_beta / n_mc

            # d/d_log_sigma = d/d_beta * d_beta/d_log_sigma
            # d_beta/d_log_sigma = sigma * epsilon = (beta_s - mu)
            grad_log_sigma += (grad_beta * (beta_s - mu)) / n_mc

        # Add entropy gradient (from -E_q[log q] = sum(log sigma) + const)
        grad_log_sigma += 1.0  # d/d_log_sigma of sum(log sigma)

        # Subtract prior KL gradient
        # KL = 0.5 * sum((sigma^2 + mu^2)/tau^2 - 1 - 2*log_sigma + log(tau^2))
        # This is already handled in the ELBO via log_prior - log_q

        # Gradient ascent
        mu += learning_rate * grad_mu
        log_sigma += learning_rate * grad_log_sigma

        elbo_history.append(elbo)

        if iteration > 100 and iteration % 50 == 0:
            recent = elbo_history[-50:]
            if abs(np.mean(recent[-25:]) - np.mean(recent[:25])) < tol:
                break

    return {
        "mu": mu,
        "sigma": np.exp(log_sigma),
        "elbo_history": elbo_history,
    }

In [ ]:
rng_vi_lr = np.random.default_rng(42)
vi_lr_result = vi_logistic_regression(X, y, tau=10.0, n_mc=20,
                                       learning_rate=0.005, max_iter=3000,
                                       rng=rng_vi_lr)

print("Variational posterior:")
for j in range(p):
    print(f"  beta_{j}: mean={vi_lr_result['mu'][j]:.4f}, "
          f"std={vi_lr_result['sigma'][j]:.4f}")
print(f"\nMAP estimate: {beta_map}")
print(f"True beta:    {beta_true}")

### Comparison with MCMC

In [ ]:
# Run MCMC for comparison (from MCMC lecture)
def metropolis_hastings(log_target, x0, proposal_sd, n_iter, rng=None):
    """Random walk Metropolis-Hastings sampler."""
    if rng is None:
        rng = np.random.default_rng()
    d = len(x0)
    samples = np.zeros((n_iter, d))
    n_accept = 0
    x_current = x0.copy()
    log_pi_current = log_target(x_current)

    for t in range(n_iter):
        x_proposed = x_current + rng.normal(0, proposal_sd, size=d)
        log_pi_proposed = log_target(x_proposed)
        if np.log(rng.uniform()) < log_pi_proposed - log_pi_current:
            x_current = x_proposed
            log_pi_current = log_pi_proposed
            n_accept += 1
        samples[t] = x_current

    return {"samples": samples, "acceptance_rate": n_accept / n_iter}


rng_mh = np.random.default_rng(42)
mcmc_result = metropolis_hastings(
    lambda b: log_posterior(b, X, y),
    x0=np.zeros(p), proposal_sd=0.15, n_iter=20000, rng=rng_mh
)
burnin = 5000
mcmc_samples = mcmc_result["samples"][burnin:]

print(f"MCMC acceptance rate: {mcmc_result['acceptance_rate']:.2%}")
print(f"MCMC posterior mean: {mcmc_samples.mean(axis=0)}")
print(f"MCMC posterior std:  {mcmc_samples.std(axis=0)}")
print(f"\nVI posterior mean: {vi_lr_result['mu']}")
print(f"VI posterior std:  {vi_lr_result['sigma']}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# ELBO convergence (smoothed)
window = 50
elbo_arr = np.array(vi_lr_result["elbo_history"])
if len(elbo_arr) > window:
    smoothed = np.convolve(elbo_arr, np.ones(window)/window, mode="valid")
    axes[0].plot(smoothed, color="steelblue", linewidth=1)
else:
    axes[0].plot(elbo_arr, color="steelblue", linewidth=1)
axes[0].set_xlabel("Iteration")
axes[0].set_ylabel("ELBO (smoothed)")
axes[0].set_title("VI Convergence")

# Posterior comparison for each beta
for j in range(p):
    ax = axes[j + 1]

    # MCMC histogram
    ax.hist(mcmc_samples[:, j], bins=50, density=True, alpha=0.5,
            color="steelblue", edgecolor="white", label="MCMC")

    # VI Gaussian
    beta_grid = np.linspace(
        vi_lr_result["mu"][j] - 4 * vi_lr_result["sigma"][j],
        vi_lr_result["mu"][j] + 4 * vi_lr_result["sigma"][j],
        200
    )
    vi_pdf = sp_stats.norm.pdf(beta_grid, vi_lr_result["mu"][j],
                                vi_lr_result["sigma"][j])
    ax.plot(beta_grid, vi_pdf, "r-", linewidth=2, label="VI")
    ax.axvline(beta_true[j], color="black", linestyle="--", label="True")
    ax.set_xlabel(f"$\\beta_{j}$")
    ax.set_ylabel("Density")
    ax.set_title(f"Posterior: $\\beta_{j}$")
    ax.legend(fontsize=8)

plt.tight_layout()

The VI Gaussian approximation captures the posterior location (mean) well but may slightly underestimate the posterior spread compared to MCMC. This is expected from the reverse KL / mean-field approximation. However, VI runs in seconds while MCMC took much longer.

### Question

The VI approximation for Bayesian logistic regression uses a factorized Gaussian: $q(\boldsymbol{\beta}) = \prod_j N(\mu_j, \sigma_j^2)$. The MCMC posterior shows some correlation between $\beta_0$ and $\beta_1$.

(a) Can the mean-field VI approximation capture this correlation? What effect does ignoring it have?

(b) Would you expect the VI posterior to over- or under-estimate the marginal variance of each $\beta_j$? Why?

### Answer

(a) No, the mean-field factorization forces $q(\beta_0)$ and $q(\beta_1)$ to be independent by construction. This means the joint variational distribution $q(\beta_0, \beta_1)$ is a product of marginals, forming an axis-aligned ellipse in the $(\beta_0, \beta_1)$ plane, while the true posterior may be a tilted ellipse. Ignoring correlation means that conditional inferences (e.g., the distribution of $\beta_1$ given a specific value of $\beta_0$) will be wrong, even though the marginal means may be correct.

(b) Mean-field VI with reverse KL typically *underestimates* marginal variances. The reverse KL penalty forces $q$ to zero wherever $p$ is small, making $q$ more concentrated than $p$. Additionally, ignoring the posterior correlation can further reduce the apparent marginal variance. For example, if the posterior has a tilted elliptical shape, the axis-aligned rectangle that fits inside it (mode-seeking) has smaller marginal spread than the ellipse itself.

## Stochastic Variational Inference

### Motivation

CAVI requires a full pass through the data to update each variational factor. For large datasets ($n$ in the millions), this is too expensive. **Stochastic variational inference (SVI)**, introduced by Hoffman et al. (2013), scales VI to large datasets by using stochastic optimization with mini-batches.

### The Idea

Instead of computing the exact ELBO gradient using all $n$ observations, SVI estimates the gradient using a random subset (mini-batch) and takes a stochastic gradient ascent step. By the law of large numbers, the mini-batch gradient is an unbiased estimate of the full gradient. Under appropriate learning rate conditions, stochastic optimization theory guarantees convergence to a stationary point of the ELBO.

### Algorithm

For a model with global parameters $\boldsymbol{\theta}$ (e.g., cluster means) and local latent variables $\mathbf{z}_i$ (e.g., cluster assignments):

```
SVI Algorithm:
1. Initialize global variational parameters λ
2. For t = 1, 2, ...:
   a. Sample mini-batch S of size B from {1, ..., n}
   b. For each i in S: update local parameters φ_i (full local update)
   c. Compute intermediate global parameters from mini-batch:
      λ_hat = optimal λ using only mini-batch (scaled to full data size n)
   d. Update global parameters: λ_t = (1 - ρ_t) λ_{t-1} + ρ_t * λ_hat
   where ρ_t is a decaying learning rate satisfying
   Σ ρ_t = ∞ and Σ ρ_t^2 < ∞ (Robbins-Monro conditions)
```

The learning rate schedule $\rho_t = (t + \tau_0)^{-\kappa}$ with $\kappa \in (0.5, 1]$ and delay $\tau_0 \geq 0$ satisfies these conditions.

### Implementation: SVI for GMM

In [ ]:
def svi_gmm(x, K, sigma0=5.0, batch_size=50, n_iter=5000,
            kappa=0.7, tau0=10, elbo_every=200, rng=None):
    """Stochastic variational inference for Gaussian mixture model.

    Parameters
    ----------
    x : array of shape (n,)
    K : int, number of components
    sigma0 : float, prior std for mu_k
    batch_size : int, mini-batch size
    n_iter : int, number of iterations
    kappa : float, learning rate decay exponent (0.5, 1]
    tau0 : float, learning rate delay
    elbo_every : int, compute full-data ELBO every this many iterations

    Returns
    -------
    dict with m, s2, elbo_history
    """
    if rng is None:
        rng = np.random.default_rng()
    n = len(x)

    # Initialize global variational parameters
    m = rng.normal(np.mean(x), np.std(x), K)
    s2 = np.ones(K)

    elbo_history = []

    for t in range(1, n_iter + 1):
        rho_t = (t + tau0) ** (-kappa)  # learning rate

        # Sample mini-batch
        batch_idx = rng.choice(n, size=batch_size, replace=False)
        x_batch = x[batch_idx]

        # Full local update for mini-batch
        phi_batch = np.zeros((batch_size, K))
        for k in range(K):
            phi_batch[:, k] = m[k] * x_batch - 0.5 * (s2[k] + m[k]**2)
        phi_batch -= phi_batch.max(axis=1, keepdims=True)
        phi_batch = np.exp(phi_batch)
        phi_batch /= phi_batch.sum(axis=1, keepdims=True)

        # Compute intermediate global parameters from mini-batch (scaled to n)
        for k in range(K):
            n_k_hat = (n / batch_size) * np.sum(phi_batch[:, k])
            s2_hat = 1.0 / (1.0 / sigma0**2 + n_k_hat)
            m_hat = s2_hat * (n / batch_size) * np.sum(phi_batch[:, k] * x_batch)

            # Robbins-Monro update
            m[k] = (1 - rho_t) * m[k] + rho_t * m_hat
            s2[k] = (1 - rho_t) * s2[k] + rho_t * s2_hat

        # Compute full-data ELBO periodically for monitoring
        if t % elbo_every == 0:
            phi_full = np.zeros((n, K))
            for k in range(K):
                phi_full[:, k] = m[k] * x - 0.5 * (s2[k] + m[k]**2)
            phi_full -= phi_full.max(axis=1, keepdims=True)
            phi_full = np.exp(phi_full)
            phi_full /= phi_full.sum(axis=1, keepdims=True)

            E_mu2 = s2 + m**2
            elbo = 0.0
            for k in range(K):
                elbo += np.sum(phi_full[:, k] * (
                    -0.5 * np.log(2 * np.pi)
                    - 0.5 * (x**2 - 2 * x * m[k] + E_mu2[k])
                ))
            for k in range(K):
                elbo += np.sum(phi_full[:, k] * (-np.log(K)))
                safe_phi = np.clip(phi_full[:, k], 1e-15, 1.0)
                elbo -= np.sum(phi_full[:, k] * np.log(safe_phi))
            for k in range(K):
                elbo += -0.5 * np.log(2 * np.pi * sigma0**2) - 0.5 * E_mu2[k] / sigma0**2
                elbo += 0.5 * np.log(2 * np.pi * np.e * s2[k])
            elbo_history.append(elbo)

    return {"m": m, "s2": s2, "elbo_history": elbo_history}

In [ ]:
# Compare CAVI (full data) vs SVI (mini-batches)
rng_cavi = np.random.default_rng(42)
rng_svi = np.random.default_rng(42)

cavi_result = cavi_gmm(x_mix, K=2, sigma0=5.0, max_iter=200, rng=rng_cavi)
svi_result = svi_gmm(x_mix, K=2, sigma0=5.0, batch_size=50, n_iter=5000, rng=rng_svi)

# Sort by mean for comparison
cavi_order = np.argsort(cavi_result["m"])
svi_order = np.argsort(svi_result["m"])

print("Cluster means:")
print(f"  True:  {mu_true_mix}")
print(f"  CAVI:  {cavi_result['m'][cavi_order]}")
print(f"  SVI:   {svi_result['m'][svi_order]}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# CAVI ELBO
axes[0].plot(cavi_result["elbo_history"], "o-", markersize=3, color="steelblue")
axes[0].set_xlabel("Iteration")
axes[0].set_ylabel("ELBO")
axes[0].set_title("CAVI Convergence (Full Data)")

# SVI ELBO (full-data ELBO evaluated periodically)
elbo_every = 200
svi_iters = range(elbo_every, elbo_every * len(svi_result["elbo_history"]) + 1,
                  elbo_every)
axes[1].plot(svi_iters, svi_result["elbo_history"], "o-", markersize=3,
             color="tab:orange")
axes[1].set_xlabel("Iteration")
axes[1].set_ylabel("ELBO (full data)")
axes[1].set_title("SVI Convergence (Mini-Batch Updates)")

plt.tight_layout()

Although SVI uses only a mini-batch per iteration, the full-data ELBO (evaluated periodically) converges to the same value as CAVI. SVI requires more iterations than CAVI because each stochastic step is noisier, but it processes far fewer data points per iteration, making it practical for large datasets where CAVI would be too slow.

### Question

In SVI, the learning rate $\rho_t = (t + \tau_0)^{-\kappa}$ must satisfy the Robbins-Monro conditions: $\sum_t \rho_t = \infty$ and $\sum_t \rho_t^2 < \infty$.

(a) Why is the first condition ($\sum \rho_t = \infty$) needed?

(b) Why is the second condition ($\sum \rho_t^2 < \infty$) needed?

(c) What happens if you use a constant learning rate $\rho_t = \rho$ that does not decay?

### Answer

(a) $\sum \rho_t = \infty$ ensures that the cumulative step sizes are large enough to reach the optimum from any starting point. If the learning rates decayed too fast (e.g., $\rho_t = 1/t^2$), the algorithm might stop making progress before reaching the optimum, getting stuck at a suboptimal point.

(b) $\sum \rho_t^2 < \infty$ ensures that the noise from the stochastic gradients eventually averages out. The variance of the stochastic updates at iteration $t$ is proportional to $\rho_t^2$. If this sum is infinite, the accumulated noise never vanishes and the iterates keep oscillating.

(c) With a constant learning rate, $\sum \rho_t^2 = \infty$, violating the second condition. The algorithm will not converge to the optimum but instead oscillate around it with variance proportional to $\rho$. In practice, constant learning rates are sometimes used with the understanding that the final solution is approximate. Averaging the iterates or reducing $\rho$ after an initial exploration phase can mitigate this.

## VI vs. MCMC

Having now seen both approaches in detail, let us systematically compare them.

### Side-by-Side Comparison

| Property | MCMC | Variational Inference |
|----------|------|----------------------|
| Nature | Sampling | Optimization |
| Guarantee | Asymptotically exact | Approximate |
| Speed | Slow (serial, correlated samples) | Fast (optimization, parallelizable) |
| Scalability | $O(n)$ per iteration, hard to mini-batch | $O(B)$ per iteration via SVI |
| Posterior quality | Full posterior (with enough samples) | Constrained by variational family |
| Variance estimation | Accurate (given mixing) | Typically underestimates |
| Multimodality | Can explore (but may mix slowly) | Usually captures one mode |
| Diagnostics | Well-established ($\hat{R}$, ESS, trace plots) | ELBO monitoring (fewer guarantees) |
| Correlations | Captured naturally | Ignored by mean-field |

### Empirical Comparison on Logistic Regression

In [ ]:
import time

# Time VI
rng_vi_time = np.random.default_rng(42)
t0 = time.time()
vi_timed = vi_logistic_regression(X, y, tau=10.0, n_mc=20,
                                   learning_rate=0.005, max_iter=3000,
                                   rng=rng_vi_time)
vi_time = time.time() - t0

# Time MCMC
rng_mh_time = np.random.default_rng(42)
t0 = time.time()
mcmc_timed = metropolis_hastings(
    lambda b: log_posterior(b, X, y),
    x0=np.zeros(p), proposal_sd=0.15, n_iter=20000, rng=rng_mh_time
)
mcmc_time = time.time() - t0

mcmc_post_burnin = mcmc_timed["samples"][5000:]

print(f"Method     | Time (s) | beta_0 mean (std)     | beta_1 mean (std)")
print(f"-" * 72)
print(f"MCMC       | {mcmc_time:7.2f}  | {mcmc_post_burnin[:,0].mean():.4f} "
      f"({mcmc_post_burnin[:,0].std():.4f})  | {mcmc_post_burnin[:,1].mean():.4f} "
      f"({mcmc_post_burnin[:,1].std():.4f})")
print(f"VI         | {vi_time:7.2f}  | {vi_timed['mu'][0]:.4f} "
      f"({vi_timed['sigma'][0]:.4f})  | {vi_timed['mu'][1]:.4f} "
      f"({vi_timed['sigma'][1]:.4f})")
print(f"\nTrue beta: {beta_true}")

### When to Use Which

**Use MCMC when:**

- You need accurate posterior uncertainty (credible intervals, tail probabilities).
- The posterior may be multimodal or have complex geometry.
- The dataset is small to moderate ($n$ up to tens of thousands).
- You need to explore the posterior thoroughly for model checking.

**Use VI when:**

- The dataset is large ($n$ in the hundreds of thousands or millions) and MCMC is too slow.
- You need to quickly compare many models or hyperparameter settings.
- Approximate posterior summaries (means, rough uncertainty) are sufficient.
- The model has a natural mean-field structure (e.g., topic models, mixture models).

**Use both when:**

- Start with VI for fast exploration and initialization, then refine with MCMC for the final analysis.
- Use VI to identify promising models, then run MCMC on the selected model.

### Question

A researcher is fitting a Bayesian hierarchical model with 500 subjects, each with 10 observations, and 5 population-level parameters. The total dataset size is 5,000 observations.

(a) Would you recommend MCMC or VI for this problem? Why?

(b) If the dataset were 100x larger (500,000 observations), would your recommendation change?

(c) The researcher wants to report 95% credible intervals for the population-level parameters. Does this affect your recommendation?

### Answer

(a) For $n = 5{,}000$, MCMC is likely the better choice. The dataset is moderate-sized, and MCMC will give exact posterior inference (including credible intervals and posterior predictive checks) in a reasonable amount of time. Gibbs sampling or HMC with the hierarchical structure would handle this well.

(b) With $n = 500{,}000$, each MCMC iteration requires evaluating the likelihood for all 500,000 observations, which could be prohibitively slow. SVI becomes attractive because it can use mini-batches. However, if the 500 subject-level random effects create strong dependencies, mean-field VI may miss important posterior structure. A practical approach would be to use SVI for the observation-level terms and more careful treatment (block updates or full-rank VI) for the population-level parameters.

(c) Yes, the need for credible intervals favors MCMC. Mean-field VI tends to underestimate posterior variance, so the 95% credible intervals from VI may have less than 95% actual coverage. If accurate uncertainty quantification is critical, MCMC is preferred, or VI should be validated against MCMC on a subset of the data before trusting the full-data VI results.

## Practical VI Software

We have implemented variational inference from scratch to understand the mechanics. In practice, probabilistic programming frameworks automate the optimization, gradient computation, and variational family specification. The key idea behind these tools is **automatic differentiation variational inference (ADVI)**: given a probabilistic model, the software automatically constructs a variational approximation and optimizes the ELBO using automatic differentiation.

The same frameworks we used for MCMC in the previous lecture, PyMC and NumPyro, also support VI. Where we previously called `pm.sample()` or `MCMC(...).run()`, we now call `pm.fit()` or `SVI(...).run()`.

### PyMC ADVI

PyMC implements ADVI through `pm.fit()`, which fits a Gaussian variational approximation by maximizing the ELBO. The model specification is identical to the MCMC case; only the inference call changes.

In [ ]:
import pymc as pm
import arviz as az

# Same logistic regression model as before
with pm.Model() as vi_logistic_model:
    beta_pm = pm.Normal("beta", mu=0, sigma=10, shape=p)
    eta = pm.math.dot(X, beta_pm)
    pm.Bernoulli("y_obs", logit_p=eta, observed=y)

    # ADVI: fit a Gaussian approximation (compare with pm.sample() for MCMC)
    approx = pm.fit(n=30000, method="advi", random_seed=42,
                    progressbar=False)

# ELBO convergence
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(-approx.hist)
axes[0].set_xlabel("Iteration")
axes[0].set_ylabel("ELBO")
axes[0].set_title("PyMC ADVI convergence")

# Draw samples from the fitted approximation and summarize
vi_trace = approx.sample(5000)
beta_samples_pm = vi_trace.posterior["beta"].values.reshape(-1, p)

axes[1].set_title("Posterior comparison")
x_offsets = np.arange(p)
width = 0.25
axes[1].bar(x_offsets - width, [vi_lr_result['mu'][j] for j in range(p)],
            width, label="Custom VI", color="steelblue")
axes[1].bar(x_offsets, beta_samples_pm.mean(axis=0), width,
            label="PyMC ADVI", color="coral")
axes[1].bar(x_offsets + width, beta_true, width,
            label="True", color="gray")
axes[1].set_xticks(x_offsets)
axes[1].set_xticklabels([f"$\\beta_{j}$" for j in range(p)])
axes[1].set_ylabel("Posterior mean")
axes[1].legend()

plt.tight_layout()

# Print summary
print("PyMC ADVI posterior:")
for j in range(p):
    print(f"  beta_{j}: mean={beta_samples_pm[:, j].mean():.4f}, "
          f"std={beta_samples_pm[:, j].std():.4f}")
print(f"\nCustom VI:")
for j in range(p):
    print(f"  beta_{j}: mean={vi_lr_result['mu'][j]:.4f}, "
          f"std={vi_lr_result['sigma'][j]:.4f}")
print(f"\nTrue beta: {beta_true}")

### NumPyro SVI

NumPyro is a JAX-based probabilistic programming library. For VI, it provides **stochastic variational inference (SVI)**, where the user specifies a model, a **guide** (the variational family), an optimizer, and a loss function (typically the ELBO). NumPyro includes automatic guides that construct the variational family from the model: `AutoNormal` produces a mean-field Gaussian (independent normals, like our custom implementation), while `AutoMultivariateNormal` produces a full-rank Gaussian that captures posterior correlations. Switching between families is a one-line change.

In [ ]:
import jax
import jax.numpy as jnp
import numpyro
import numpyro.distributions as dist
from numpyro.infer import SVI, Trace_ELBO
from numpyro.infer.autoguide import AutoNormal, AutoMultivariateNormal
from numpyro.optim import Adam

def logistic_model_numpyro(X, y=None):
    beta = numpyro.sample("beta", dist.Normal(jnp.zeros(X.shape[1]), 10.0))
    logits = X @ beta
    numpyro.sample("y_obs", dist.Bernoulli(logits=logits), obs=y)

X_jnp, y_jnp = jnp.array(X), jnp.array(y)

# Mean-field VI (AutoNormal) -- same assumption as our custom implementation
guide_mf = AutoNormal(logistic_model_numpyro)
svi_mf = SVI(logistic_model_numpyro, guide_mf, Adam(0.01), Trace_ELBO())
svi_mf_result = svi_mf.run(jax.random.PRNGKey(42), 10000,
                           X=X_jnp, y=y_jnp, progress_bar=False)

# Full-rank VI (AutoMultivariateNormal) -- captures posterior correlations
guide_fr = AutoMultivariateNormal(logistic_model_numpyro)
svi_fr = SVI(logistic_model_numpyro, guide_fr, Adam(0.01), Trace_ELBO())
svi_fr_result = svi_fr.run(jax.random.PRNGKey(42), 10000,
                           X=X_jnp, y=y_jnp, progress_bar=False)

# Extract posterior samples
mf_samples = guide_mf.sample_posterior(
    jax.random.PRNGKey(0), svi_mf_result.params, sample_shape=(5000,)
)["beta"]
fr_samples = guide_fr.sample_posterior(
    jax.random.PRNGKey(0), svi_fr_result.params, sample_shape=(5000,)
)["beta"]

# Plot ELBO convergence and compare mean-field vs full-rank
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(svi_mf_result.losses, label="Mean-field", alpha=0.7)
axes[0].plot(svi_fr_result.losses, label="Full-rank", alpha=0.7)
axes[0].set_xlabel("Iteration")
axes[0].set_ylabel("ELBO loss")
axes[0].set_title("NumPyro SVI convergence")
axes[0].legend()

axes[1].set_title("Mean-field vs. full-rank")
x_offsets = np.arange(p)
width = 0.2
axes[1].bar(x_offsets - 1.5*width, [vi_lr_result['mu'][j] for j in range(p)],
            width, label="Custom VI", color="steelblue")
axes[1].bar(x_offsets - 0.5*width, np.array(mf_samples.mean(axis=0)),
            width, label="NumPyro mean-field", color="coral")
axes[1].bar(x_offsets + 0.5*width, np.array(fr_samples.mean(axis=0)),
            width, label="NumPyro full-rank", color="gold")
axes[1].bar(x_offsets + 1.5*width, beta_true, width,
            label="True", color="gray")
axes[1].set_xticks(x_offsets)
axes[1].set_xticklabels([f"$\\beta_{j}$" for j in range(p)])
axes[1].set_ylabel("Posterior mean")
axes[1].legend()

plt.tight_layout()

print("NumPyro SVI posterior:")
print("  Mean-field guide (AutoNormal):")
for j in range(p):
    print(f"    beta_{j}: mean={float(mf_samples[:, j].mean()):.4f}, "
          f"std={float(mf_samples[:, j].std()):.4f}")
print("  Full-rank guide (AutoMultivariateNormal):")
for j in range(p):
    print(f"    beta_{j}: mean={float(fr_samples[:, j].mean()):.4f}, "
          f"std={float(fr_samples[:, j].std()):.4f}")
print(f"\nTrue beta: {beta_true}")

### Software Comparison for VI

| Feature | PyMC (`pm.fit`) | NumPyro (`SVI`) |
|---------|----------------|------------------|
| Variational families | Mean-field, full-rank | Mean-field, full-rank, custom, normalizing flows |
| Optimization | ADVI (automatic) | User-specified optimizer |
| Speed | Moderate | Fast (JAX, GPU support) |
| Flexibility | Limited to built-in methods | Full control over guide and loss |
| Learning curve | Low (`pm.fit()` is one line) | Moderate (guide + optimizer setup) |

Our custom implementation of mean-field VI for logistic regression took about 50 lines of NumPy code for the ELBO, gradients, and optimization loop. With PyMC, the same result requires about 5 lines (model specification + `pm.fit()`). With NumPyro, choosing between mean-field and full-rank guides is a one-line change (`AutoNormal` vs. `AutoMultivariateNormal`), replacing the manual parameterization we built by hand.

### Question

A researcher wants to fit a Bayesian neural network with 10,000 parameters on a dataset of 1 million images.

(a) Would you recommend MCMC or VI? Why?

(b) Should the researcher use a mean-field or full-rank variational family?

(c) The researcher plans to first fit the model with VI, then run MCMC on a 1% subset of the data to validate the VI approximation. Is this a reasonable workflow?

### Answer

(a) VI is the clear choice. MCMC with 10,000 parameters would be extremely slow: even HMC scales as $O(d^{5/4})$ per independent sample, and evaluating the likelihood on 1 million images per iteration is prohibitively expensive. SVI with mini-batches makes this problem tractable.

(b) Mean-field. A full-rank Gaussian over 10,000 parameters requires parameterizing a $10{,}000 \times 10{,}000$ covariance via its Cholesky factor ($O(d^2) = 10^8$ variational parameters, about 400 MB of storage). Mean-field needs only $2d = 20{,}000$ variational parameters. For neural networks, the posterior correlations are typically less important than getting reasonable marginal uncertainty for each weight.

(c) This is a reasonable validation strategy with one caveat. Running MCMC on a 1% subset (10,000 images) with 10,000 parameters may be feasible for simpler architectures but remains expensive. Additionally, the posterior on the subset will differ from the full-data posterior. The researcher should compare VI and MCMC results on the same subset, not compare subset-MCMC to full-data-VI. If VI and MCMC agree on the subset, it provides evidence that VI is a reasonable approximation for this model class.

## Summary

Variational inference provides an optimization-based alternative to MCMC for approximate Bayesian inference. The key ideas are:

1. **The ELBO** provides a tractable lower bound on the log-evidence. Maximizing the ELBO is equivalent to minimizing the KL divergence from the variational approximation to the true posterior. The ELBO decomposes into an expected log-likelihood (data fit) and a KL penalty (regularization toward the prior).

2. **KL directionality** matters. VI minimizes the reverse KL $\text{KL}(q \| p)$, which is mode-seeking: the approximation tends to concentrate on the dominant mode and underestimate posterior variance. This contrasts with the forward KL, which would produce mass-covering approximations.

3. **Mean-field VI** assumes the variational distribution factorizes across parameters. The optimal factors have a closed form in exponential family models, leading to the CAVI algorithm. CAVI cycles through parameters like Gibbs sampling but uses optimization rather than sampling.

4. **Stochastic variational inference** scales VI to large datasets by replacing full-data ELBO computations with mini-batch estimates. With proper learning rate decay (Robbins-Monro conditions), SVI converges to a stationary point of the ELBO while processing only a subset of data per iteration.

### Connections to Prior Lectures

- **EM algorithm:** EM is a special case of variational inference where $q(\mathbf{Z})$ is set to the exact conditional distribution and parameters are optimized as point estimates. Full VI generalizes this by approximating the posterior over all unknowns.

- **Numerical integration:** The ELBO avoids computing the marginal likelihood $p(\mathbf{y}) = \int p(\mathbf{y} | \boldsymbol{\theta}) p(\boldsymbol{\theta}) d\boldsymbol{\theta}$ directly. The Laplace approximation from the numerical integration lecture can be viewed as a special case of VI where $q$ is Gaussian and centered at the posterior mode.

- **MCMC:** Both MCMC and VI target the posterior, but through different mechanisms. MCMC generates (correlated) samples from the exact posterior. VI finds the closest tractable approximation. They are complementary: VI is faster, MCMC is more accurate. The comparison table in this lecture provides guidance on when to use each.

### References

- Blei, D. M., Kucukelbir, A., & McAuliffe, J. D. (2017). Variational inference: A review for statisticians. *Journal of the American Statistical Association*, 112(518), 859-877.
- Jordan, M. I., Ghahramani, Z., Jaakkola, T. S., & Saul, L. K. (1999). An introduction to variational methods for graphical models. *Machine Learning*, 37(2), 183-233.
- Hoffman, M. D., Blei, D. M., Wang, C., & Paisley, J. (2013). Stochastic variational inference. *Journal of Machine Learning Research*, 14, 1303-1347.
- Kingma, D. P., & Welling, M. (2014). Auto-encoding variational Bayes. In *Proceedings of the 2nd International Conference on Learning Representations (ICLR)*.
- Bishop, C. M. (2006). *Pattern Recognition and Machine Learning*. Springer. Chapter 10.